# OpenCV: Webcam-Farbkanäle, Graustufen und Kantenerkennung

Dieses Notebook vereint alle gewünschten Ansichten in **einem Dropdown**:
- **Rot**: Pro Pixel bleibt nur der Rotanteil erhalten (in Rot dargestellt).
- **Grün**: Pro Pixel bleibt nur der Grünanteil erhalten (in Grün dargestellt).
- **Blau**: Pro Pixel bleibt nur der Blauanteil erhalten (in Blau dargestellt).
- **Graustufen**: Die Farbinformation wird entfernt; jeder Pixel wird durch seine Helligkeit ersetzt.
- **Kanten (Canny)**: Erkennt Helligkeitsübergänge und zeigt sie als helle Linien auf schwarzem Hintergrund.

Zusätzlich gibt es **Original**. Alle Filter können während des Webcam-Livestreams über dasselbe Dropdown ausgewählt werden. Läuft auch ohne Webcam (automatisches Testbild).


## 1. Installation und Imports

In [ ]:
# Nur beim ersten Mal (Kommentar entfernen):
# %pip install opencv-python numpy matplotlib ipywidgets

In [ ]:
import sys, time, threading
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image as IPyImage

print("OpenCV Version:", cv2.__version__)

## 2. Bildquelle: Webcam oder Testbild

In [ ]:
USE_WEBCAM = True    # False = immer Testbild
CAM_INDEX = 0        # bei mehreren Kameras 1, 2, ... probieren
MIRROR = True        # Spiegelbild


def testbild(t=0, w=640, h=480):
    img = np.zeros((h, w, 3), np.uint8)
    for y in range(h):
        img[y, :] = (60 + y * 60 // h, 60 + y * 50 // h, 60)
    dx = int(40 * np.sin(t / 10))
    cv2.circle(img, (140 + dx, 150), 60, (0, 0, 220), -1)                 # rot (BGR)
    cv2.rectangle(img, (320, 90 - dx), (450, 220 - dx), (0, 200, 0), -1)  # gruen
    pts = np.array([[500, 330], [580 + dx, 420], [430, 420]])
    cv2.fillPoly(img, [pts], (220, 60, 0))                                # blau
    cv2.circle(img, (160, 360), 50, (0, 220, 230), -1)                    # gelb
    cv2.rectangle(img, (300, 320), (380, 400), (255, 255, 255), -1)       # weiss
    cv2.putText(img, "Testbild", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    return img


def open_cam(index=0):
    try:
        if sys.platform.startswith("win"):
            cap = cv2.VideoCapture(index, cv2.CAP_DSHOW)
        else:
            cap = cv2.VideoCapture(index)
        if cap.isOpened():
            ok, _ = cap.read()
            if ok:
                return cap
        cap.release()
    except Exception as e:
        print("Webcam-Fehler:", e)
    return None


class Quelle:
    def __init__(self, webcam=USE_WEBCAM, index=CAM_INDEX):
        self.cap = open_cam(index) if webcam else None
        self.t = 0
        print("Quelle:", "Webcam" if self.cap else "Testbild (keine Webcam gefunden/aktiv)")

    def read(self):
        if self.cap is not None:
            ok, frame = self.cap.read()
            if not ok:
                return None
            return cv2.flip(frame, 1) if MIRROR else frame
        self.t += 1
        return testbild(self.t)

    def release(self):
        if self.cap is not None:
            self.cap.release()
            self.cap = None


def snapshot():
    q = Quelle()
    if q.cap:
        time.sleep(0.5)
    frame = None
    for _ in range(5):          # erste Frames sind oft dunkel
        frame = q.read()
    q.release()
    return frame

## 3. Farbkanäle und Kantenerkennung

OpenCV speichert Bilder als **BGR** (Blau, Grün, Rot). Jeder Kanal enthält pro Pixel einen Wert von 0 bis 255.

Die drei Farbfilter übernehmen jeweils nur einen Kanal und setzen die beiden anderen auf 0. Es wird kein Pixel aussortiert.

Der zusätzliche **Canny-Kantenfilter** wandelt das Bild zuerst in Graustufen um, glättet leichtes Bildrauschen und erkennt anschließend Kanten. Er zeigt Kanten weiß auf schwarzem Hintergrund.


In [ ]:
def kanal_rot(f):
    return f[:, :, 2]

def kanal_gruen(f):
    return f[:, :, 1]

def kanal_blau(f):
    return f[:, :, 0]


# OpenCV verwendet BGR: Rot=(0,0,255), Grün=(0,255,0), Blau=(255,0,0)
KANALE = {
    "rot":   (kanal_rot,   (0, 0, 255)),
    "gruen": (kanal_gruen, (0, 255, 0)),
    "blau":  (kanal_blau,  (255, 0, 0)),
}

def filtere(f, name):
    """Behält pro Pixel nur den gewählten Farbkanal und zeigt ihn in seiner Farbe."""
    funktion, farbe = KANALE[name]
    kanal = funktion(f).astype(np.float32) / 255.0
    out = np.zeros_like(f, dtype=np.float32)
    for i in range(3):
        out[:, :, i] = kanal * farbe[i]
    return out.astype(np.uint8)

def filtere_graustufen(f):
    """Entfernt die Farbinformation und zeigt die Helligkeit jedes Pixels in Grau."""
    grau = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(grau, cv2.COLOR_GRAY2BGR)

def kanten_canny(f, low=50, high=150):
    """Erkennt Kanten und gibt sie weiß auf schwarzem Hintergrund zurück."""
    grau = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
    weich = cv2.GaussianBlur(grau, (5, 5), 0)
    kanten = cv2.Canny(weich, low, high)
    return cv2.cvtColor(kanten, cv2.COLOR_GRAY2BGR)

def wende_an(f, name):
    if name == "original":
        return f
    if name == "graustufen":
        return filtere_graustufen(f)
    if name == "kanten (Canny)":
        return kanten_canny(f)
    return filtere(f, name)


## 4. Einzelbild: Original, Farbkanäle, Graustufen und Kanten

Alle Ansichten werden aus demselben Webcam-Bild berechnet. Die Graustufenansicht ist ein eigener Filter und steht ebenso wie Rot, Grün, Blau und Canny im Live-Dropdown zur Verfügung.


In [ ]:
frame = snapshot()

def uebersicht(frame):
    namen = ["rot", "gruen", "blau", "graustufen", "kanten (Canny)"]
    fig, axs = plt.subplots(2, 3, figsize=(15, 8))
    axs = axs.ravel()
    axs[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axs[0].set_title("Original")
    for ax, n in zip(axs[1:], namen):
        ax.imshow(cv2.cvtColor(wende_an(frame, n), cv2.COLOR_BGR2RGB))
        ax.set_title(n.capitalize())
    for ax in axs:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

uebersicht(frame)


## 5. Live-Stream mit Dropdown

Der Stream läuft in einem Hintergrund-Thread. Du kannst während des Streams zwischen **Original, Rot, Grün, Blau, Graustufen und Canny-Kanten** wechseln.


In [ ]:
def jpeg(img, qualitaet=80):
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, qualitaet])
    return buf.tobytes()

bild = widgets.Image(format="jpeg", width=640)
auswahl = widgets.Dropdown(options=["original", "rot", "gruen", "blau", "graustufen", "kanten (Canny)"],
                           value="rot", description="Filter:")
btn_start = widgets.Button(description="Start", button_style="success")
btn_stop = widgets.Button(description="Stopp", button_style="danger")
status = widgets.Label("bereit")
state = {"run": False, "thread": None}

def stream_loop():
    q = Quelle()
    status.value = "laeuft"
    fps_t, fps_n = time.time(), 0
    try:
        while state["run"]:
            f = q.read()
            if f is None:
                status.value = "kein Kamerabild"
                break
            out = wende_an(f, auswahl.value)
            fps_n += 1
            dt = time.time() - fps_t
            if dt >= 1:
                status.value = f"laeuft, {fps_n / dt:.0f} FPS"
                fps_t, fps_n = time.time(), 0
            bild.value = jpeg(out)
            time.sleep(0.01)
    finally:
        q.release()
        state["run"] = False
        status.value = "gestoppt"

def on_start(_):
    if state["thread"] is not None and state["thread"].is_alive():
        return
    state["run"] = True
    state["thread"] = threading.Thread(target=stream_loop, daemon=True)
    state["thread"].start()

def on_stop(_):
    state["run"] = False

btn_start.on_click(on_start)
btn_stop.on_click(on_stop)
display(widgets.VBox([widgets.HBox([btn_start, btn_stop, auswahl, status]), bild]))
